# 17 — Bagging via K-Fold CV (CatBoost, categóricas nativas)
## PRT Seguros

Em vez de um único split treino/validação (80/20), treinamos **5 modelos CatBoost**, cada um numa
fatia diferente (5-fold CV) usando **as 100 mil linhas rotuladas inteiras** (treino + validação
juntos). Cada modelo prevê no Kaggle, e a submissão final é a **média das 5 previsões** —
técnica clássica de competição (bagging) que costuma generalizar melhor do que um modelo único,
porque cada fold vê uma fatia diferente dos dados e os erros individuais tendem a se cancelar
na média.

Como bônus, a métrica *out-of-fold* (cada linha avaliada pelo modelo que NÃO a viu no treino)
usa as 100 mil linhas inteiras — uma estimativa de AUC mais estável que um único split de 20 mil.

## 1. Carregar e reconstruir categóricas (igual ao notebook `16`)

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready.csv")
val = pd.read_csv("dados_processados/val_model_ready.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready.csv")

GRUPOS_CATEGORICOS = {
    "escolaridade": ["escolaridade_Fundamental", "escolaridade_Medio", "escolaridade_NaN",
                      "escolaridade_Pos", "escolaridade_Superior"],
    "canal_aquisicao": ["canal_aquisicao_Agente", "canal_aquisicao_Digital", "canal_aquisicao_Indicacao",
                         "canal_aquisicao_NaN", "canal_aquisicao_Telefone"],
    "canal": ["canal_Carta", "canal_Email", "canal_Nao Informado", "canal_Telefone", "canal_WhatsApp"],
    "metodo_pagamento": ["metodo_pagamento_NaN", "metodo_pagamento_boleto", "metodo_pagamento_credito",
                          "metodo_pagamento_debito", "metodo_pagamento_pix"],
    "tipo_cobertura": ["tipo_cobertura_NaN", "tipo_cobertura_basica", "tipo_cobertura_padrao",
                        "tipo_cobertura_premium"],
    "segmento": ["segmento_Bronze", "segmento_Diamante", "segmento_NaN", "segmento_Ouro", "segmento_Prata"],
    "veiculo": ["veiculo_Carro", "veiculo_Moto", "veiculo_NaN", "veiculo_Pickup", "veiculo_SUV", "veiculo_Van"],
    "regiao": ["regiao_Centro-Oeste", "regiao_NaN", "regiao_Nordeste", "regiao_Sudeste", "regiao_Sul"],
}

def reconstruir_categoricas(df):
    df = df.copy()
    for grupo, cols in GRUPOS_CATEGORICOS.items():
        prefixo = grupo + "_"
        df[grupo] = df[cols].idxmax(axis=1).str.replace(prefixo, "", regex=False)
        df = df.drop(columns=cols)
    df["cluster"] = df["cluster"].astype(int).astype(str)
    return df

treino_completo = pd.concat([train, val], ignore_index=True)
treino_completo = reconstruir_categoricas(treino_completo)
kaggle_cat = reconstruir_categoricas(kaggle)

COLUNAS_CATEGORICAS = list(GRUPOS_CATEGORICOS.keys()) + ["cluster"]
feature_cols = [c for c in treino_completo.columns if c not in (ID_COL, TARGET_COL)]
cat_idx = [feature_cols.index(c) for c in COLUNAS_CATEGORICAS]

print(f"Treino completo (treino+val): {treino_completo.shape}")
print(f"Kaggle: {kaggle_cat.shape}")


Treino completo (treino+val): (100000, 42)
Kaggle: (100000, 41)


## 2. Treinar 5 folds e coletar previsões out-of-fold + previsões no Kaggle

In [2]:
MELHORES_PARAMS_CAT = {"random_strength": 0.5, "learning_rate": 0.02, "l2_leaf_reg": 9, "depth": 6}

X_full = treino_completo[feature_cols]
y_full = treino_completo[TARGET_COL]
X_kaggle = kaggle_cat[feature_cols]
pool_kaggle = Pool(X_kaggle, cat_features=cat_idx)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_proba = np.zeros(len(X_full))
kaggle_proba_por_fold = []

for fold, (idx_tr, idx_val_fold) in enumerate(skf.split(X_full, y_full)):
    X_tr_fold, y_tr_fold = X_full.iloc[idx_tr], y_full.iloc[idx_tr]
    X_val_fold, y_val_fold = X_full.iloc[idx_val_fold], y_full.iloc[idx_val_fold]

    # separa uma fatia do treino do fold só pra early stopping (nunca usa o val_fold pra isso)
    idx_es_tr, idx_es_val = train_test_split(
        np.arange(len(X_tr_fold)), test_size=0.15, stratify=y_tr_fold, random_state=RANDOM_STATE
    )
    neg_pos_ratio = (y_tr_fold.iloc[idx_es_tr] == 0).sum() / (y_tr_fold.iloc[idx_es_tr] == 1).sum()

    pool_tr = Pool(X_tr_fold.iloc[idx_es_tr], y_tr_fold.iloc[idx_es_tr], cat_features=cat_idx)
    pool_es = Pool(X_tr_fold.iloc[idx_es_val], y_tr_fold.iloc[idx_es_val], cat_features=cat_idx)
    pool_val_fold = Pool(X_val_fold, cat_features=cat_idx)

    modelo_fold = CatBoostClassifier(
        iterations=3000, **MELHORES_PARAMS_CAT, scale_pos_weight=neg_pos_ratio,
        eval_metric="AUC", random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
    )
    modelo_fold.fit(pool_tr, eval_set=pool_es)

    proba_fold = modelo_fold.predict_proba(pool_val_fold)[:, 1]
    oof_proba[idx_val_fold] = proba_fold
    auc_fold = roc_auc_score(y_val_fold, proba_fold)

    proba_kaggle_fold = modelo_fold.predict_proba(pool_kaggle)[:, 1]
    kaggle_proba_por_fold.append(proba_kaggle_fold)

    print(f"Fold {fold+1}/5 — melhor iteração: {modelo_fold.get_best_iteration():>4}  AUC-ROC = {auc_fold:.4f}")

auc_oof = roc_auc_score(y_full, oof_proba)
print(f"\nAUC-ROC out-of-fold (100 mil linhas): {auc_oof:.4f}")
print(f"Referência (split único, notebook 16): 0.8264")


Fold 1/5 — melhor iteração:  276  AUC-ROC = 0.8315


Fold 2/5 — melhor iteração:  255  AUC-ROC = 0.8268


Fold 3/5 — melhor iteração:  448  AUC-ROC = 0.8298


Fold 4/5 — melhor iteração:  586  AUC-ROC = 0.8240


Fold 5/5 — melhor iteração:  207  AUC-ROC = 0.8178

AUC-ROC out-of-fold (100 mil linhas): 0.8259
Referência (split único, notebook 16): 0.8264


## 3. Gerar submissão com a média dos 5 folds (bagging)

In [3]:
proba_kaggle_bagging = np.mean(kaggle_proba_por_fold, axis=0)

submissao = pd.DataFrame({"Id": kaggle_cat[ID_COL], "probabilidade_churn": proba_kaggle_bagging})
submissao.to_csv("submissions/submission_bagging_5fold.csv", index=False)
print(f"AUC-ROC out-of-fold: {auc_oof:.4f}")
print("Salvo: submissions/submission_bagging_5fold.csv")
submissao.head()


AUC-ROC out-of-fold: 0.8259
Salvo: submissions/submission_bagging_5fold.csv


,Id,probabilidade_churn
0,221300000002,0.115256
1,221300000020,0.225689
2,221300000097,0.166978
3,221300000148,0.418311
4,221300000166,0.321448
